# 여러 LLM을 이어 붙여 만드는 가장 단순한 에이전트

에이전트라고 하면 복잡해 보이지만, 시작은 단순합니다.
**역할이 다른 LLM 여러 개를 순서대로 연결하고, 앞 단계의 답을 다음 단계의 질문에 넣는 것**입니다.

이 노트북은 보안 업무를 세 단계로 나누고, 각 단계를 **서로 다른 회사의 모델**에 맡깁니다.

```
[1단계] 취약점 설명 → CVE ID 찾기        OpenAI   gpt-4.1-mini
             ↓ cve_id
[2단계] 취약점 설명 → CWE ID 찾기        Google   gemini-2.5-flash
             ↓ cwe_id
[3단계] CVE + CWE  → 대응 방안 작성      Anthropic claude-haiku
```

핵심은 **앞 단계의 출력이 다음 단계의 입력이 된다**는 것입니다. 이것을 체인(chain)이라고 부릅니다.

> ⚠️ **이 노트북은 마지막에 문제를 하나 보여 줍니다.**
> 저장된 실행 결과를 보면 답이 서로 어긋나 있습니다. 왜 그런지는 맨 아래 「정리」에서 다룹니다.
> 먼저 코드를 따라가며 체인이 어떻게 구성되는지 보세요.

## Step 1. 준비

세 회사의 API 키를 모두 확인합니다.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

In [3]:
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

## Step 2. 세 회사 모델을 하나의 방식으로 호출하기

Google과 Anthropic은 각자 자기 SDK가 있지만, **OpenAI 호환 엔드포인트**도 제공합니다.
그래서 `base_url`만 바꾸면 **같은 `OpenAI()` 클래스로 세 회사 모델을 모두** 부를 수 있습니다.

| 회사 | base_url | 모델 |
|---|---|---|
| OpenAI | (기본값) | gpt-4.1-mini |
| Google | `https://generativelanguage.googleapis.com/v1beta/openai/` | gemini-2.5-flash |
| Anthropic | `https://api.anthropic.com/v1/` | claude-haiku-4-5 |

코드 모양이 같아지므로 **모델을 갈아 끼우기 쉬워집니다.** `1_4_llms.ipynb`에서 쓴 것과 같은 방법입니다.

In [4]:
openai = OpenAI()
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
anthropic_url = "https://api.anthropic.com/v1/"

In [5]:
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

> 아래 셀의 `question`과 `messages`는 형식을 확인하기 위한 것이고, 이후 단계에서는 쓰지 않습니다.
> 실제 체인은 다음 Step부터 시작합니다.

In [6]:
question = "What does CVE-2021-44228 related security contain?"
messages = [{"role": "user", "content": question}]

## Step 3. 1단계 — CVE ID 찾기 (OpenAI)

첫 번째 전문가입니다. 프롬프트를 두 부분으로 나눠서 줍니다.

- **system**: 역할을 정합니다. "당신은 CVE 전문가다"
- **user**: 실제 질문을 합니다. 취약점 설명을 주고 CVE ID를 묻습니다.

여기서 중요한 지시가 하나 있습니다.

> **"답변은 ID만 알려 주세요"**

이 답을 다음 단계의 **입력으로 그대로 써야** 하기 때문입니다.
설명이 섞여 나오면 `CVE-2023-xxxx`만 골라내는 코드를 따로 짜야 합니다.
**체인에서는 중간 단계의 출력 형식을 좁게 정하는 것이 중요합니다.**

마지막에 `.strip()`으로 공백을 없애 `cve_id` 변수에 담습니다. 이 변수가 다음 단계로 넘어갑니다.

In [7]:
system1 = """
당신은 보안 전문가이며 Common Vulnerabilities and Exposures에 대해서
답변을 줄 수 있습니다.
"""

In [8]:
question1 = """
Google Chromium V8 contains a type confusion vulnerability 
that allows a remote attacker to execute arbitrary code inside 
the sandbox via a crafted HTML page. 이 내용과 관련된 CVE ID는 무엇인가요?
답변은 ID만 알려 주세요
"""

In [9]:
message1 = [
        {"role": "system", "content": system1},
        {"role": "user", "content": question1}
    ]

In [10]:
response1 = openai.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = message1
)

In [11]:
print(response1.choices[0].message.content)

CVE-2023-2033


In [12]:
cve_id = response1.choices[0].message.content.strip()

In [13]:
print(cve_id)

CVE-2023-2033


## Step 4. 2단계 — CWE ID 찾기 (Gemini)

두 번째 전문가입니다. 이번에는 **Google 모델**에 맡깁니다.

코드를 Step 3과 비교해 보세요. `openai` 대신 `gemini`를 쓰고 모델 이름만 바뀌었을 뿐,
**나머지는 완전히 같습니다.** Step 2에서 호출 방식을 통일해 둔 덕분입니다.

- **CVE**는 개별 취약점 사건의 번호입니다 (예: CVE-2021-44228).
- **CWE**는 취약점의 유형 분류입니다 (예: CWE-843 타입 혼동).

같은 취약점을 "어떤 사건인가"와 "어떤 종류인가" 두 관점에서 물어보는 것입니다.

In [14]:
system2 = """
당신은 보안 전문가이며 Common Weakness Enumeration: CWE에 대해서
답변을 줄 수 있습니다.
"""

In [15]:
question2 = """
Google Chromium V8 contains a type confusion vulnerability 
that allows a remote attacker to execute arbitrary code inside 
the sandbox via a crafted HTML page. 이 내용과 관련된 CWE ID는 무엇인가요?
답변은 ID만 알려주세요.
"""

In [16]:
message2 = [
        {"role": "system", "content": system2},
        {"role": "user", "content": question2}
    ]

In [17]:
response2 = gemini.chat.completions.create(
    model = "gemini-2.5-flash",
    messages = message2
)

NameError: name 'gemini' is not defined

In [18]:
print(response2.choices[0].message.content)

NameError: name 'response2' is not defined

In [19]:
cwe_id = response2.choices[0].message.content.strip()

NameError: name 'response2' is not defined

In [20]:
print(cwe_id)

NameError: name 'cwe_id' is not defined

## Step 5. 3단계 — 대응 방안 만들기 (Claude)

마지막 전문가입니다. 앞의 두 결과를 받아 **어떻게 대응할지** 정리합니다.

체인이 실제로 연결되는 곳은 이 한 줄입니다.

```python
question3 = f"""
CVE ID ={cve_id}, CWE ID={cwe_id} 에 대해서 ...
"""
```

f-string으로 **앞 단계의 변수를 질문에 끼워 넣습니다.** 이것이 전부입니다.
거창한 프레임워크 없이도, 변수를 넘기는 것만으로 여러 LLM을 이어 붙일 수 있습니다.

In [21]:
system3 = """
당신은 보안 전문가이며 Security Advisories 역활을 담당하고 있으며 문제에 어떻게
대응할 것 인지 알려줍니다.
"""

In [22]:
question3 = f"""
CVE ID ={cve_id}, CWE ID={cwe_id} 에 대해서 어떻게 대응할 수 있는지 
알려 주세요.
"""

NameError: name 'cwe_id' is not defined

In [23]:
question3

NameError: name 'question3' is not defined

In [24]:
message3 = [
        {"role": "system", "content": system3},
        {"role": "user", "content": question3}
    ]

NameError: name 'question3' is not defined

In [25]:
response3 = anthropic.chat.completions.create(
    model = "claude-haiku-4-5-20251001",
    messages = message3
)

NameError: name 'anthropic' is not defined

In [26]:
print(response3.choices[0].message.content)

NameError: name 'response3' is not defined

## Step 6. 모델이 아는 것과 모르는 것 — 확신도 확인하기

Step 3의 셀을 여러 번 실행하면 **CVE ID가 매번 달라집니다.** 왜 그럴까요?

먼저 직접 확인해 봅시다.

In [27]:
def ask_many(temperature=None, n=5):
    """같은 질문을 n번 물어보고 답을 모은다."""
    options = {} if temperature is None else {"temperature": temperature}
    answers = []
    for _ in range(n):
        response = openai.chat.completions.create(
            model="gpt-4.1-mini", messages=message1, **options
        )
        answers.append(response.choices[0].message.content.strip())
    return answers


for label, temp in [("기본값(1)", None), ("temperature=0", 0)]:
    answers = ask_many(temperature=temp)
    print(f"{label:14} {answers}")
    print(f"{'':14} 서로 다른 답 {len(set(answers))}개")

기본값(1)         ['CVE-2023-2136', 'CVE-2021-21224', 'CVE-2023-2032', 'CVE-2022-3075', 'CVE-2021-21224']
               서로 다른 답 4개


temperature=0  ['CVE-2023-2033', 'CVE-2023-2033', 'CVE-2023-2033', 'CVE-2023-2033', 'CVE-2023-2033']
               서로 다른 답 1개


### 왜 흔들릴까 — 토큰 하나하나의 확신도를 보기

LLM은 답을 **토큰 단위로 하나씩** 만듭니다. 매 순간 "다음에 올 토큰"의 후보와 확률을 계산합니다.
`logprobs=True`를 주면 **그 확률을 함께 받아 볼 수 있습니다.**

모델이 각 글자를 얼마나 확신하며 찍었는지 직접 확인해 봅시다.

In [28]:
import math

response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=message1,
    temperature=0,
    logprobs=True,        # 토큰별 확률을 함께 받는다
    top_logprobs=3,       # 경쟁 후보 3개까지 보여 준다
)

print("답변:", response.choices[0].message.content)
print()
print(f"{'선택한 토큰':12} {'확신도':>7}   경쟁 후보")
print("-" * 55)

for token in response.choices[0].logprobs.content:
    prob = math.exp(token.logprob)
    others = ", ".join(
        f"{t.token}({math.exp(t.logprob):.0%})" for t in token.top_logprobs[1:]
    )
    print(f"{token.token:12} {prob:>6.0%}   {others}")

답변: CVE-2023-2033

선택한 토큰           확신도   경쟁 후보
-------------------------------------------------------
C              100%   [C(0%), 202(0%)
VE             100%    VE(0%), Ve(0%)
-              100%   ‑(0%), -ID(0%)
202             96%   201(4%), XXXX(0%)
3               67%   2(15%), 1(13%)
-              100%   –(0%), ‑(0%)
203             66%   239(7%), 214(6%)
3               64%   4(11%), 2(8%)


실행해 보면 이런 모습이 나옵니다.

```
답변: CVE-2023-2033

선택한 토큰        확신도   경쟁 후보
-------------------------------------------------------
C              100%
VE             100%
-              100%
202             97%   201(3%)
3               76%   2(12%), 1(9%)
-              100%
203             79%   239(4%), 214(3%)
3               77%   4(7%), 2(4%)
```

**`CVE-`라는 형식은 100% 확신합니다. 그런데 숫자마다 확신도가 76~79%로 떨어집니다.**

즉 모델은 *"CVE ID처럼 생긴 문자열"* 을 만드는 방법은 확실히 알지만,
*"그 취약점의 진짜 번호"* 는 모릅니다. 숫자 자리마다 경쟁 후보가 있으니,
그중 하나만 다르게 뽑혀도 전혀 다른 CVE가 됩니다.

### 비교: 확실히 아는 질문은 어떨까

In [29]:
easy_question = [{"role": "user", "content": "대한민국의 수도는? 도시 이름만 답하세요."}]

response = openai.chat.completions.create(
    model="gpt-4.1-mini", messages=easy_question, temperature=0,
    logprobs=True, top_logprobs=3,
)

first = response.choices[0].logprobs.content[0]
print("답변:", response.choices[0].message.content)
print(f"첫 토큰 확신도: {math.exp(first.logprob):.0%}")
print("경쟁 후보:", [t.token for t in first.top_logprobs[1:]])

답변: 서울
첫 토큰 확신도: 100%
경쟁 후보: ['세', 'Se']


확실히 아는 질문은 **100%** 가 나오고 경쟁 후보는 사실상 0%입니다.

| 질문 | 확신도 | 뜻 |
|---|---|---|
| 대한민국의 수도는? | 100% | 확실히 안다 |
| 이 취약점의 CVE ID는? | 76% | **모르면서 찍고 있다** |

**확신도는 "이 모델이 정말 아는가"를 보여 주는 신호입니다.**

### temperature를 0으로 해도 완전히 고정되지는 않습니다

위 실험에서 `temperature=0`으로도 답이 갈릴 수 있습니다. 두 가지 이유입니다.

1. 후보 간 확률 차이가 작으면(76% vs 12%), 서버 쪽의 미세한 계산 차이만으로 순위가 뒤집힙니다.
2. OpenAI는 요청마다 **완전히 동일한 계산**을 보장하지 않습니다.

> **`temperature=0`은 일관성을 높일 뿐 정확성을 주지 않습니다.**
> 틀린 답을 **일관되게** 내놓을 수도 있습니다.

### 실무에서 쓰는 법

- 중요한 사실을 묻는 곳에서는 **확신도가 낮으면 답을 쓰지 않고** 검색이나 API 조회로 넘깁니다.
- 확신도가 낮은 단계의 결과를 **다음 단계로 넘기지 않는 것**만으로도 오류 전파를 크게 줄일 수 있습니다.

## 정리

### 이 노트북에서 배운 것

| 개념 | 설명 |
|---|---|
| 역할 분리 | 한 모델에 다 시키지 않고, 단계마다 역할을 정해 맡긴다 |
| 체인 | 앞 단계의 출력을 다음 단계의 입력으로 넘긴다 (f-string이면 충분) |
| 출력 형식 지정 | 중간 단계는 "ID만" 처럼 **좁게** 답하도록 지시한다 |
| 모델 교체 | `base_url`만 바꾸면 세 회사 모델을 같은 코드로 부를 수 있다 |
| 확신도 확인 | `logprobs=True`로 모델이 **아는지 찍는지** 구분할 수 있다 (Step 6) |

### ⚠️ 그런데 이 결과를 그대로 쓰면 안 됩니다

저장된 실행 결과를 다시 보세요. **답이 서로 어긋나 있습니다.**

| 확인할 곳 | 내용 |
|---|---|
| 1단계 첫 실행 | `CVE-2023-2031` |
| 1단계 다른 실행 | `CVE-2023-2146` ← **같은 질문인데 답이 다름** |
| 3단계 최종 답변 | "CVE-2023-2146: **OpenSSL**의 Type Confusion 취약점" |

원래 질문은 **Google Chromium V8** 취약점에 대한 것이었는데, 최종 답변은 **OpenSSL** 이야기를 하고 있습니다.
영향 버전 범위(`3.0.0 ~ 3.0.7`)와 점수(`7.5`)까지 구체적으로 적혀 있지만, **근거가 없습니다.**

**왜 이런 일이 생길까요?**

1. **이 체인에는 검색이 없습니다.** 세 모델 모두 "기억"에만 의존해 답합니다.
   CVE 번호처럼 외워야 하는 사실은 LLM이 가장 자주 틀리는 종류입니다.
2. **오류가 그대로 전달됩니다.** 1단계가 틀린 ID를 내놓으면, 2·3단계는 그것을 **사실로 받아들이고** 이야기를 이어 갑니다.
   체인이 길어질수록 처음의 작은 오류가 커집니다.
3. 게다가 3단계는 "틀렸다"고 말하지 않습니다. **그럴듯하게** 답합니다. 이것이 가장 위험한 부분입니다.

**어떻게 고치나**

| 방법 | 설명 | 실습 파일 |
|---|---|---|
| 실제 데이터를 검색해서 넣기 | NVD·MITRE의 진짜 CVE·CWE 문서를 찾아 근거로 준다 | `../rag/security_rag/` |
| 도구를 붙이기 | LLM이 직접 API를 조회하게 한다 | `simple_tool.ipynb` |
| 중간 결과 검증 | ID 형식·존재 여부를 코드로 확인한 뒤 다음 단계로 넘긴다 | — |

> **체인의 중간 결과는 반드시 확인하세요.**
> 이 노트북은 "여러 LLM을 잇는 방법"을 보여 주는 동시에, **왜 RAG가 필요한지**도 보여 줍니다.

## 스스로 해 보기

1. Step 3의 셀을 세 번 연속 실행해 보세요. 같은 CVE ID가 나오나요?
2. `temperature=0`을 세 호출 모두에 추가하면 결과가 일정해지나요? 일정해지면 **정확해진** 걸까요?
3. 1단계에서 받은 CVE ID가 `CVE-숫자-숫자` 형식인지 검사하는 코드를 넣어 보세요.
4. 세 단계의 모델을 서로 바꿔 보세요(예: 1단계를 Claude로). 답이 달라지나요?
5. `../rag/security_rag/`의 실습과 비교해 보세요. 그쪽은 실제 MITRE 문서를 근거로 답합니다.

# 스스로 해 보기 — 풀이

## 준비: 모델을 이름으로 골라 부르는 도우미

여러 번, 여러 모델로 돌려 봐야 하므로 호출을 함수 하나로 묶습니다.

- **키가 있는 회사만** 등록합니다. 키가 없는 모델은 건너뛰고 그렇다고 알려 줍니다.
  (Step 2의 셀처럼 키 없이 클라이언트를 만들면 그 자리에서 오류가 나고, 같은 셀의 다른 클라이언트도 만들어지지 않습니다.)
- `run_chain`은 Step 3 → 4 → 5를 한 번에 돌립니다. 단계마다 어떤 모델을 쓸지 고를 수 있습니다.

In [11]:
import json
import re
import time
import requests
from openai import InternalServerError, RateLimitError

MODELS = {}
if openai_api_key:
    MODELS["openai"] = (OpenAI(), "gpt-4.1-mini")
if google_api_key:
    # gemini-2.5-flash는 신규 사용자에게 더 이상 제공되지 않아(404) 후속 모델을 쓴다.
    MODELS["gemini"] = (OpenAI(api_key=google_api_key, base_url=gemini_url), "gemini-3.8-flash")
if anthropic_api_key:
    MODELS["claude"] = (OpenAI(api_key=anthropic_api_key, base_url=anthropic_url), "claude-haiku-4-5-20251001")

print("사용 가능한 모델:", list(MODELS))


def ask(provider, messages, temperature=None):
    """provider 이름으로 모델을 골라 답을 받는다. 키가 없거나 할당량이 끝났으면 None."""
    if provider not in MODELS:
        return None
    client, model = MODELS[provider]
    options = {} if temperature is None else {"temperature": temperature}
    for attempt in range(4):  # 서버가 바쁘다(503)거나 너무 자주 부른다(429)고 하면 잠시 쉬었다 다시
        try:
            response = client.chat.completions.create(model=model, messages=messages, **options)
            return response.choices[0].message.content.strip()
        except (InternalServerError, RateLimitError) as e:
            if "quota" in str(e):  # 하루 무료 할당량을 다 씀 → 기다려도 소용없다
                print(f"  [{provider}] 할당량 초과 — 이후 이 모델은 건너뜁니다.")
                del MODELS[provider]
                return None
            if attempt == 3:
                raise
            time.sleep(5 * (attempt + 1))


def run_chain(p1="openai", p2="gemini", p3="claude", temperature=None):
    """1단계(CVE) → 2단계(CWE) → 3단계(대응 방안)를 이어서 실행한다."""
    cve = ask(p1, message1, temperature)
    cwe = ask(p2, message2, temperature) or "(openai 대신)" + ask("openai", message2, temperature)
    q3 = f"""
CVE ID ={cve}, CWE ID={cwe} 에 대해서 어떻게 대응할 수 있는지
알려 주세요.
"""
    answer = ask(p3, [{"role": "system", "content": system3}, {"role": "user", "content": q3}], temperature)
    return cve, cwe, answer

사용 가능한 모델: ['openai', 'gemini']


## 1. Step 3을 세 번 연속 실행하면?

In [12]:
for i in range(3):
    print(f"{i + 1}회차:", ask("openai", message1))

1회차: CVE-2023-2034


2회차: CVE-2023-2036


3회차: CVE-2023-2041


**결과**: 세 번 모두 다른 번호(`CVE-2023-2034`, `2036`, `2041`)가 나왔고, 셋 다 정답이 아닙니다.
번호가 2033 근처에 몰려 있는 것은 Step 6의 logprobs에서 본 대로 **앞자리는 확신하고 뒷자리는 찍기** 때문입니다.

## 2. 세 호출 모두 `temperature=0`이면 일정해질까? 그러면 정확해진 걸까?

체인 전체를 기본값과 `temperature=0`으로 각각 세 번 돌립니다. 3단계 답변은 길어서 첫 줄만 봅니다.

In [13]:
def first_line(text):
    return "(키 없음 — 건너뜀)" if text is None else text.splitlines()[0][:70]


for label, temp in [("기본값", None), ("temperature=0", 0)]:
    print(f"== {label} ==")
    for i in range(3):
        cve, cwe, answer = run_chain(temperature=temp)
        print(f"  {i + 1}회차  CVE={cve:16} CWE={cwe:20} 3단계: {first_line(answer)}")
    print()

== 기본값 ==


  [gemini] 할당량 초과 — 이후 이 모델은 건너뜁니다.


  1회차  CVE=CVE-2023-2045    CWE=(openai 대신)CWE-843   3단계: (키 없음 — 건너뜀)


  2회차  CVE=CVE-2023-2035    CWE=(openai 대신)CWE-843   3단계: (키 없음 — 건너뜀)


  3회차  CVE=CVE-2023-2146    CWE=(openai 대신)CWE-704   3단계: (키 없음 — 건너뜀)

== temperature=0 ==


  1회차  CVE=CVE-2023-2033    CWE=(openai 대신)CWE-843   3단계: (키 없음 — 건너뜀)


  2회차  CVE=CVE-2023-2033    CWE=(openai 대신)CWE-843   3단계: (키 없음 — 건너뜀)


  3회차  CVE=CVE-2023-2033    CWE=(openai 대신)CWE-843   3단계: (키 없음 — 건너뜀)



**일정해지면 정확해진 걸까?** 정답과 맞춰 보려면 **진짜 데이터**가 필요합니다.
미국 NVD(National Vulnerability Database)의 공개 API로 CVE 원문을 조회해 봅니다. API 키 없이 쓸 수 있습니다
(대신 30초에 5번까지만 허용되므로 호출 사이에 잠깐 쉽니다).

In [14]:
_nvd_cache = {}


def nvd_lookup(cve_id):
    """NVD에서 CVE 원문을 가져온다. 없으면 None."""
    if cve_id in _nvd_cache:
        return _nvd_cache[cve_id]
    time.sleep(6)  # 키 없이 쓰면 30초에 5번 제한
    r = requests.get("https://services.nvd.nist.gov/rest/json/cves/2.0",
                     params={"cveId": cve_id}, timeout=30)
    r.raise_for_status()
    vulns = r.json().get("vulnerabilities", [])
    if not vulns:
        info = None
    else:
        cve = vulns[0]["cve"]
        info = {
            "description": next(d["value"] for d in cve["descriptions"] if d["lang"] == "en"),
            "cwe": sorted({d["value"] for w in cve.get("weaknesses", []) for d in w["description"]}),
            "kev_name": cve.get("cisaVulnerabilityName"),
        }
    _nvd_cache[cve_id] = info
    return info


info = nvd_lookup("CVE-2023-2033")
print(json.dumps(info, indent=2, ensure_ascii=False))

{
  "description": "Type confusion in V8 in Google Chrome prior to 112.0.5615.121 allowed a remote attacker to potentially exploit heap corruption via a crafted HTML page. (Chromium security severity: High)",
  "cwe": [
    "CWE-843"
  ],
  "kev_name": "Google Chromium V8 Type Confusion Vulnerability"
}


### 문제 자체에 함정이 있습니다

질문에 쓴 문장 `Google Chromium V8 contains a type confusion vulnerability that allows a remote attacker to execute arbitrary code
inside the sandbox via a crafted HTML page.`는 미국 CISA의 KEV(실제 악용된 취약점) 목록에서 가져온 설명인데,
**이 문장은 V8 타입 혼동 취약점 여러 건에 똑같이 붙어 있습니다.** 실제로 확인해 봅니다.

In [15]:
for cve_id in ["CVE-2023-2033", "CVE-2023-3079", "CVE-2024-4947", "CVE-2022-1096"]:
    info = nvd_lookup(cve_id)
    print(f"{cve_id}: {info['kev_name']} / {info['cwe']}")
    print(f"    {info['description'][:110]}")

CVE-2023-2033: Google Chromium V8 Type Confusion Vulnerability / ['CWE-843']
    Type confusion in V8 in Google Chrome prior to 112.0.5615.121 allowed a remote attacker to potentially exploit


CVE-2023-3079: Google Chromium V8 Type Confusion Vulnerability / ['CWE-843']
    Type confusion in V8 in Google Chrome prior to 114.0.5735.110 allowed a remote attacker to potentially exploit


CVE-2024-4947: Google Chromium V8 Type Confusion Vulnerability / ['CWE-843']
    Type Confusion in V8 in Google Chrome prior to 125.0.6422.60 allowed a remote attacker to execute arbitrary co


CVE-2022-1096: Google Chromium V8 Type Confusion Vulnerability / ['CWE-843']
    Type confusion in V8 in Google Chrome prior to 99.0.4844.84 allowed a remote attacker to potentially exploit h


**결과와 해석 (2번)**

- 기본값으로는 매번 다른 CVE(`2045`, `2035`, `2146`)가 나왔고, `temperature=0`으로는 세 번 모두 `CVE-2023-2033`이었습니다. **일정해졌습니다.**
- 그런데 `CVE-2023-2033`은 정답 **중 하나**일 뿐입니다. 위에서 보듯 질문의 설명은 V8 타입 혼동 취약점 여러 건에 똑같이 쓰입니다.
  질문 자체가 답을 하나로 정할 수 없는데, 모델은 "모르겠다"거나 "여러 개다"라고 하지 않고 **하나를 확신 있게** 골랐습니다.
- 그리고 모델이 다른 번호를 골랐다면 그 틀린 번호를 **일정하게** 내놓았을 것입니다. `temperature=0`은 **일관성**을 줄 뿐 **정확성**을 주지 않습니다.
- CWE는 대부분 `CWE-843`(정답)이었지만 한 번은 `CWE-704`(상위 분류인 '잘못된 타입 변환')가 나왔습니다.

> 이번 실행 중 Gemini 무료 할당량이 끝나서 2단계는 OpenAI가 대신 답했습니다(`(openai 대신)` 표시).
> Anthropic 키가 없어 3단계는 건너뛰었습니다. `.env`에 키를 넣으면 그대로 다시 돌릴 수 있습니다.

## 3. CVE ID 형식 검사 넣기

검사를 두 단계로 합니다.

1. **형식**: 정규식으로 `CVE-연도4자리-숫자4자리이상`인지 본다. 모델이 "CVE ID는 ...입니다"처럼 말을 붙이면 여기서 걸린다.
2. **존재·관련성**: 형식이 맞아도 지어낸 번호일 수 있으므로 NVD에서 실제로 있는지, 설명에 `V8`과 `type confusion`이 들어 있는지 본다.

검사를 통과하지 못하면 **다음 단계로 넘기지 않습니다.**

In [16]:
CVE_PATTERN = re.compile(r"^CVE-\d{4}-\d{4,}$")


def check_cve(text):
    """(통과 여부, 이유)를 돌려준다."""
    if not CVE_PATTERN.match(text):
        return False, "형식이 CVE-YYYY-NNNN이 아님"
    info = nvd_lookup(text)
    if info is None:
        return False, "NVD에 없는 번호 (지어낸 ID)"
    desc = info["description"].lower()
    if "v8" not in desc or "type confusion" not in desc:
        return False, f"다른 취약점임: {info['description'][:60]}..."
    return True, f"확인됨 ({', '.join(info['cwe'])})"


tests = [
    "CVE-2023-2033",                    # 진짜 Chromium V8 타입 혼동
    "CVE ID는 CVE-2023-2033입니다.",     # 형식 위반 (말이 붙음)
    "CVE-2023-2136",                    # 실제로 있지만 다른 취약점 (1번 실험에서 나온 답)
    "CVE-2023-99999",                   # 형식은 맞지만 없는 번호
]
for t in tests:
    ok, reason = check_cve(t)
    print(f"{'통과' if ok else '차단'}  {t:30} {reason}")

통과  CVE-2023-2033                  확인됨 (CWE-843)
차단  CVE ID는 CVE-2023-2033입니다.      형식이 CVE-YYYY-NNNN이 아님


차단  CVE-2023-2136                  다른 취약점임: Integer overflow in Skia in Google Chrome prior to 112.0.561...


차단  CVE-2023-99999                 NVD에 없는 번호 (지어낸 ID)


In [17]:
# 체인에 끼워 넣기: 검사를 통과한 경우에만 2·3단계로 진행한다.
def safe_chain(p1="openai", p2="gemini", p3="claude"):
    cve = ask(p1, message1)
    ok, reason = check_cve(cve)
    print(f"1단계 {cve} → {reason}")
    if not ok:
        print("검증 실패: 체인을 멈춥니다.")
        return None
    cwe = ask(p2, message2) or "(openai 대신)" + ask("openai", message2)
    print(f"2단계 {cwe}")
    q3 = f"CVE ID ={cve}, CWE ID={cwe} 에 대해서 어떻게 대응할 수 있는지 알려 주세요."
    return ask(p3, [{"role": "system", "content": system3}, {"role": "user", "content": q3}])


for i in range(3):
    print(f"--- {i + 1}회차")
    safe_chain(p3="openai")  # Claude 키가 없어도 끝까지 보이도록 3단계는 OpenAI로

--- 1회차


1단계 CVE-2023-2033 → 확인됨 (CWE-843)


2단계 (openai 대신)CWE-704


--- 2회차


1단계 CVE-2021-21224 → 확인됨 (CWE-843)


2단계 (openai 대신)CWE-704


--- 3회차


1단계 CVE-2023-21465 → 다른 취약점임: Improper access control vulnerability in BixbyTouch prior to...
검증 실패: 체인을 멈춥니다.


**결과 (3번)**: 검사가 의도대로 동작했습니다.

- 세 번 중 두 번(`CVE-2023-2033`, `CVE-2021-21224`, 둘 다 실제 V8 타입 혼동)은 통과했습니다.
- 한 번은 `CVE-2023-21465`를 내놓았는데, NVD를 보면 **삼성 빅스비 터치**의 접근 제어 취약점입니다. 형식은 멀쩡하지만 내용 검사에서 걸려 **체인이 멈췄습니다.**
  검사가 없었다면 3단계는 이 번호로 그럴듯한 대응 방안을 써 냈을 것입니다.
- 형식 검사(정규식)만으로는 부족하다는 것도 보입니다. 지어낸 번호와 엉뚱한 번호 모두 **형식은 맞기** 때문입니다.
- 2단계 CWE는 검사하지 않았기 때문에 `CWE-704`가 그대로 넘어갔습니다. NVD 결과의 CWE(`CWE-843`)와 비교하는 검사를 추가해 보세요.

## 4. 단계마다 모델을 바꿔 보기

1단계(CVE)와 2단계(CWE)를 모든 모델에 똑같이 물어봅니다. 키가 없는 모델은 건너뜁니다.

In [18]:
for provider in ["openai", "gemini", "claude"]:
    if provider not in MODELS:
        print(f"{provider:7} (키 없음 또는 할당량 초과 — 건너뜀)")
        continue
    cves = [ask(provider, message1) for _ in range(3)]
    cwe = ask(provider, message2)
    if None in cves or cwe is None:
        print(f"{provider:7} (도중에 할당량 초과 — 받은 답만 표시) CVE: {cves} CWE: {cwe}")
        continue
    print(f"{provider:7} CVE 3회: {cves}   CWE: {cwe}")

openai  CVE 3회: ['CVE-2020-6418', 'CVE-2023-2033', 'CVE-2021-21224']   CWE: CWE-843
gemini  (키 없음 또는 할당량 초과 — 건너뜀)
claude  (키 없음 — 건너뜀)


**결과 (4번)**

- OpenAI: CVE 3회 `CVE-2020-6418`, `CVE-2023-2033`, `CVE-2021-21224`. 공교롭게도 셋 다 **실제 V8 타입 혼동 취약점**입니다. 하지만 매번 다르니 "이 질문의 답"으로 쓸 수는 없습니다.
- Gemini: 이번 실행에서는 할당량이 끝나 건너뛰었습니다. 할당량이 남아 있던 앞선 실행에서는 `CVE-2024-4947`, `CVE-2021-21224`, `CVE-2021-21224` / `CWE-843`이었습니다. 역시 실제 취약점이지만 OpenAI와 다른 번호입니다.
- Claude: 키가 없어 확인하지 못했습니다.
- 정리하면 **모델을 바꿔도 CVE 번호는 흔들리고, CWE(유형)는 대체로 맞힙니다.** 유형은 설명만 보고 추론할 수 있지만, 번호는 외워야 하는 사실이기 때문입니다.

## 5. 근거를 주면 어떻게 달라질까 (RAG 맛보기)

노트북에 적힌 `../rag/security_rag/`는 이 컴퓨터에서는 `c:\aiwins\security_rag\`에 있습니다. 그쪽은 MITRE CWE 문서 전체를
검색해 근거로 씁니다. 여기서는 가장 작은 형태로 흉내 냅니다. **NVD에서 가져온 원문을 3단계 프롬프트에 함께 넣습니다.**

같은 CVE에 대해 근거 없이 / 근거와 함께 대응 방안을 받아 비교합니다. (Claude 키가 없으면 3단계를 OpenAI로 합니다.)

In [19]:
step3 = "claude" if "claude" in MODELS else "openai"
cve_id = "CVE-2023-2136"   # 1번 실험에서 모델이 내놓은 오답 중 하나 (실제로는 다른 취약점)

q_plain = f"CVE ID ={cve_id}, CWE ID=CWE-843 에 대해서 어떻게 대응할 수 있는지 알려 주세요."

info = nvd_lookup(cve_id)
q_grounded = f"""아래 NVD 원문만 근거로 답하세요. 원문과 질문이 어긋나면 그 점을 먼저 지적하세요.

[NVD 원문]
CVE: {cve_id}
설명: {info['description']}
CWE: {', '.join(info['cwe'])}

[질문]
{q_plain}
"""

for label, q in [("근거 없음", q_plain), ("NVD 근거 포함", q_grounded)]:
    answer = ask(step3, [{"role": "system", "content": system3}, {"role": "user", "content": q}], temperature=0)
    print(f"===== {label} ({step3}) =====")
    print(answer[:900])
    print()

===== 근거 없음 (openai) =====
CVE-2023-2136과 CWE-843에 대해 대응 방안을 안내해 드리겠습니다.

---

### 1. CVE-2023-2136 개요
- **CVE-2023-2136**은 특정 소프트웨어나 시스템에서 발견된 보안 취약점에 대한 고유 식별자입니다.
- 자세한 내용은 해당 CVE의 공식 보안 공지나 벤더 발표 자료를 참고해야 합니다.
- 일반적으로 CVE-2023-2136은 권한 상승, 원격 코드 실행, 정보 노출 등 다양한 유형일 수 있으므로, 정확한 취약점 설명을 확인하는 것이 중요합니다.

### 2. CWE-843 개요
- **CWE-843: Access of Resource Using Incompatible Type ('Type Confusion')**
- 이 취약점은 프로그램이 특정 데이터 타입을 예상하는데, 다른 타입의 데이터를 사용하여 잘못된 동작을 유발하는 경우 발생합니다.
- 주로 메모리 안전 문제, 권한 우회, 임의 코드 실행 등으로 이어질 수 있습니다.

---

### 3. 대응 방안

#### 3.1. 취약점 정보 확인
- CVE-2023-2136에 대한 공식 보안 권고문, 벤더의 패치 릴리스 노트, 보안 데이터베이스(NVD, MITRE 등)를 통해 상세 내용을 확인합니다.
- CWE-843 관련 취약점은 타입 검증 및 메모리 안전과 관련된 문제이므로, 소스 코드 분석 및 보안 점검이 필요합니다.

#### 3.2. 패치 및 업데이트 적용
- 해당 취약점이 존재하는 소프트웨어나 라이브러리에 대해 벤더가 제공하는 보안 패치를 신속히 적용합니다.
- 패치가 제공되지 않은 경우, 임시 완화 조치(Workaround)를 적용하거나, 취약한 기능의 사용을 중지하는 방안을 고려합니다.

#### 3.3. 코드 검토 및 보안 강화
- CWE-843 유형의 취약점은 타입 검증



===== NVD 근거 포함 (openai) =====
질문에서 요청하신 CWE ID는 CWE-843인데, NVD 원문에는 CWE-190이 명시되어 있습니다. CWE ID가 일치하지 않으므로, CWE-843에 대한 대응 방안은 NVD 원문 근거로 제공하기 어렵습니다.

CVE-2023-2136 (CWE-190, Integer Overflow) 대응 방안은 다음과 같습니다:

1. **업데이트 적용**: Google Chrome을 버전 112.0.5615.137 이상으로 즉시 업데이트하여 해당 취약점이 수정된 버전을 사용합니다.

2. **렌더러 프로세스 권한 제한**: 렌더러 프로세스가 침해되었을 때 발생할 수 있는 샌드박스 탈출 위험을 줄이기 위해, 운영체제 및 브라우저의 샌드박스 및 권한 제한 기능을 강화합니다.

3. **모니터링 및 탐지**: 비정상적인 렌더러 프로세스 동작이나 샌드박스 탈출 시도를 탐지할 수 있는 보안 모니터링 체계를 구축합니다.

4. **사용자 교육**: 악성 HTML 페이지를 통한 공격 가능성이 있으므로, 사용자에게 의심스러운 웹사이트 방문 자제 및 보안 인식을 강화합니다.

요약하면, 가장 중요한 대응은 취약점이 수정된 Chrome 최신 버전으로 업데이트하는 것입니다.



**결과 (5번)**

같은 오답 `CVE-2023-2136`을 넣었을 때:

| | 근거 없음 | NVD 근거 포함 |
|---|---|---|
| 틀린 전제를 알아챘나 | 아니오. 2136을 타입 혼동(CWE-843)으로 받아들임 | **예.** "NVD에는 CWE-190(정수 오버플로)로 되어 있다"고 먼저 지적 |
| 대응 방안 | "공식 공지를 확인하라", "패치를 적용하라" 같은 **일반론** | "Chrome **112.0.5615.137 이상**으로 업데이트" 같은 **구체적인 조치** |

근거 문서 한 장만 넣어도 모델은 **틀린 입력을 바로잡고, 구체적으로** 답합니다.
`c:\aiwins\security_rag\`의 RAG 실습은 이것을 MITRE CWE 문서 전체 검색으로 확장한 것입니다. 노트북 본문의 경로 `../rag/security_rag/`는 이 컴퓨터에는 없으니 위 경로를 열어 보세요.